# NeoN Python coverage: benchmark `laplacian(gamma, U) - div(phi, U)`

This notebook now tracks every major element in the C++ benchmark snippet:
1. executor sweep and size sweep metadata,
2. `nCells = 10` mesh and `U`, `phi`, `gamma` setup,
3. unoptimized and fused (optimized) expression paths,
4. runtime scheme tokens (`Gauss`, `linear`, `uncorrected`, `upwind`),
5. benchmark-style loops over executors and sizes.

> Current Python bindings expose operator construction but not `Expression.read(...)`, `Expression.assemble(...)`, or `dsl::optimize(...)`. This notebook explicitly demonstrates parity where available and marks the non-exposed steps.

## 1) Imports and runtime init

In [1]:
import os
import sys
from pathlib import Path
import numpy as np

def _find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "CMakeLists.txt").exists() and (candidate / "doc" / "notebooks").exists():
            return candidate
    return None

cwd = Path(os.getcwd()).resolve()
repo_root = _find_repo_root(cwd)
if repo_root is None:
    fallback = Path("/home/andrei2/Downloads/NeoN").resolve()
    if (fallback / "CMakeLists.txt").exists():
        repo_root = fallback
    else:
        raise RuntimeError("Could not locate NeoN repo root from notebook working directory")

local_bindings = repo_root / "build" / "develop" / "bindings"
if not local_bindings.exists():
    raise RuntimeError(f"Local bindings folder not found: {local_bindings}")

if str(local_bindings) in sys.path:
    sys.path.remove(str(local_bindings))
sys.path.insert(0, str(local_bindings))

for mod_name in [m for m in list(sys.modules) if m == "neon" or m.startswith("neon.")]:
    del sys.modules[mod_name]

import neon
from neon import imp, exp

if not globals().get("_neon_initialized", False):
    neon.initialize()
    _neon_initialized = True

exec = neon.SerialExecutor()
mesh = neon.create_1d_uniform_mesh(exec, 8)

print("cwd:", cwd)
print("repo_root:", repo_root)
print("local_bindings:", local_bindings)
print("neon module:", neon.__file__)
print("executor:", exec.name())
print("cells:", mesh.n_cells())

cwd: /Volumes/Data/Code/NeoN/doc/notebooks
repo_root: /Volumes/Data/Code/NeoN
local_bindings: /Volumes/Data/Code/NeoN/build/develop/bindings
neon module: /Volumes/Data/Code/NeoN/build/develop/bindings/neon/__init__.py
executor: SerialExecutor
cells: 8


## 2) Benchmark metadata: sizes, epsilon, and available executors

In [2]:
epsilon = 1e-32
sizes = [1 << 16, 1 << 17, 1 << 18, 1 << 19, 1 << 20]

executors = [("SerialExecutor", neon.SerialExecutor()), ("CPUExecutor", neon.CPUExecutor())]
if neon.gpu_available():
    executors.append(("GPUExecutor", neon.GPUExecutor()))

print("epsilon:", epsilon)
print("sizes:", sizes)
print("executors:", [name for name, _ in executors])

epsilon: 1e-32
sizes: [65536, 131072, 262144, 524288, 1048576]
executors: ['SerialExecutor', 'CPUExecutor']


## 3) Mirror C++ field setup (`nCells = 10`, `nFaces = 9`)

In [4]:
exec_name, exec = executors[0]
n_cells = 10
mesh = neon.create_1d_uniform_mesh(exec, n_cells)

vol_bcs = neon.create_calculated_volume_bcs_scalar(mesh)
surf_bcs = neon.create_calculated_surface_bcs_scalar(mesh)

U = neon.ScalarVolumeField(exec, "U", mesh)
phi = neon.ScalarSurfaceField(exec, "phi", mesh)
gamma = neon.ScalarSurfaceField(exec, "gamma", mesh)

neon.fill(U.internal_vector(), 2.0)
neon.fill(phi.internal_vector(), 1.0)
neon.fill(gamma.internal_vector(), 2.0)

n_faces = mesh.n_internal_faces() + mesh.n_boundary_faces()
print("executor:", exec_name)
print("nCells:", mesh.n_cells())
print("nFaces (mesh):", n_faces, "(benchmark constant is 9)")
print("volume BC count:", len(vol_bcs), "surface BC count:", len(surf_bcs))


executor: SerialExecutor
nCells: 10
nFaces (mesh): 11 (benchmark constant is 9)
volume BC count: 2 surface BC count: 2


## 4) Unoptimized expression path (`laplacian - div`)

In [5]:
lap_op = imp.laplacian(gamma, U)
div_op = imp.div(phi, U)
expr_unoptimized = lap_op - div_op

print("laplacian operator:", lap_op.get_name())
print("divergence operator:", div_op.get_name())
print("unoptimized expression terms:", expr_unoptimized.size())

laplacian operator: LaplacianOperator
divergence operator: DivOperator
unoptimized expression terms: 2


## 5) Fused path mapping (`dsl::optimize(expr)` in C++)

In [6]:
expr_fused = expr_unoptimized
optimize_exposed = hasattr(neon, "optimize")

print("Python exposes neon.optimize:", optimize_exposed)
if optimize_exposed:
    expr_fused = neon.optimize(expr_unoptimized)
    print("fused expression terms:", expr_fused.size())
else:
    print("using expr_unoptimized as fused placeholder (binding not exposed yet)")


Python exposes neon.optimize: False
using expr_unoptimized as fused placeholder (binding not exposed yet)


## 6) Scheme-token coverage (`Gauss`, `linear`, `uncorrected`, `upwind`)

In [7]:
lap_tokens = neon.TokenList()
lap_tokens.insert_string("Gauss")
lap_tokens.insert_string("linear")
lap_tokens.insert_string("uncorrected")

div_tokens = neon.TokenList()
div_tokens.insert_string("Gauss")
div_tokens.insert_string("upwind")

print("laplacian(gamma,U) tokens:", [lap_tokens.get_string(i) for i in range(lap_tokens.size())])
print("div(phi,U) tokens:", [div_tokens.get_string(i) for i in range(div_tokens.size())])

print("read(Dictionary) coverage note: nested Dictionary->TokenList scheme trees are not exposed in Python yet")


laplacian(gamma,U) tokens: ['Gauss', 'linear', 'uncorrected']
div(phi,U) tokens: ['Gauss', 'upwind']
read(Dictionary) coverage note: nested Dictionary->TokenList scheme trees are not exposed in Python yet


## 7) Benchmark-style loop coverage (executors x sizes)

In [8]:
rows = []
for exec_name, exec in executors:
    mesh = neon.create_1d_uniform_mesh(exec, 10)
    U = neon.ScalarVolumeField(exec, "U", mesh)
    phi = neon.ScalarSurfaceField(exec, "phi", mesh)
    gamma = neon.ScalarSurfaceField(exec, "gamma", mesh)
    neon.fill(U.internal_vector(), 2.0)
    neon.fill(phi.internal_vector(), 1.0)
    neon.fill(gamma.internal_vector(), 2.0)

    expr_unopt = imp.laplacian(gamma, U) - imp.div(phi, U)
    expr_fused = neon.optimize(expr_unopt) if hasattr(neon, "optimize") else expr_unopt

    for size in sizes:
        rows.append((exec_name, size, expr_unopt.size(), expr_fused.size()))

print("rows produced:", len(rows))
print("first 5 rows:")
for row in rows[:5]:
    print(row)

print("assemble coverage note: Expression.assemble(...) is now exposed (with DB-registered fields)")


rows produced: 10
first 5 rows:
('SerialExecutor', 65536, 2, 2)
('SerialExecutor', 131072, 2, 2)
('SerialExecutor', 262144, 2, 2)
('SerialExecutor', 524288, 2, 2)
('SerialExecutor', 1048576, 2, 2)
assemble coverage note: Expression.assemble(...) is now exposed (with DB-registered fields)


## 8) New: implicit assembly from Python (`register_volume_field` + `assemble`)

In [9]:
exec = neon.SerialExecutor()
mesh = neon.create_1d_uniform_mesh(exec, 10)

phi = neon.ScalarVolumeField(exec, "phi", mesh)
neon.fill(phi.internal_vector(), 1.0)

db = neon.Database()
phi_reg = neon.register_volume_field(db, "fields", phi)

eq = imp.source(phi_reg, phi_reg) + imp.source(phi_reg, phi_reg)
sparsity, linear_system = eq.assemble(mesh, 0.0, 1.0)

print("expression terms:", eq.size())
print("sparsity rows:", sparsity.rows())
print("sparsity nnz:", sparsity.nnz())
print("linear system type:", type(linear_system).__name__)


expression terms: 2
sparsity rows: 10
sparsity nnz: 28
linear system type: LinearSystemScalar


## 9) Optional finalize

In [10]:
if globals().get("_neon_initialized", False):
    neon.finalize()
    _neon_initialized = False
    print("NeoN finalized")
else:
    print("NeoN already finalized")


NeoN finalized
